# Exercise 1: Analytical vs numerical forward modelling (12 min)

For a homogeneous half-space, the central-loop dB/dt response has a closed-form solution (Ward & Hohmann, 1988). For any layered earth it does not, and pyTEM computes it numerically: a Hankel transform over wavenumber, then a Fourier (DLF) or inverse Laplace (Euler) transform to time.

The half-space is the test case: if the numerical code reproduces the analytical solution there, we can trust it where no analytical solution exists.

### Functions you will need
```python
pytem.fwd_analytical_central(resistivity, tx_radius, times)          # half-space, closed form
pytem.fwd_circle_central(thicknesses, resistivities, tx_radius, times,
                         transform='dlf',            # or 'euler'
                         hankel_filter='key_101',    # or 'key_201'
                         fourier_filter='key_81')    # or 'key_101'
pytem.dbdt_to_apprho(dbdt, tx_moment, times)         # tx_moment = loop area x current [A m²]
pytem.plot_model(thicknesses, resistivities, ax=ax, label='...')
```
- `resistivities` has N entries, `thicknesses` N-1 (the last layer is the half-space). A half-space is `thicknesses=[]`, `resistivities=[rho]`.
- All forward functions return dB/dt in V/m² for 1 A. The step-off response is **negative**, so plot `-d` on log axes.

In [ ]:
# pip install pytem
import time
import numpy as np
import matplotlib.pyplot as plt
import pytem

a = np.sqrt(40**2 / np.pi)        # radius of a circle with the area of a 40 x 40 m loop [m]
times = np.logspace(-6, -2, 41)   # gate times [s]
rho = 100.0                       # half-space resistivity [Ohm.m]

### Task 1.1: does the numerical model match?
1. Compute the analytical response and the numerical response with `transform='dlf'` and with `transform='euler'`.
2. Plot all three as -dB/dt against time on log-log axes (label the y-axis in V/m²).
3. In a second panel, plot the relative error of each numerical response in % against time.

How large is the error, and where does it grow? Compare it with a typical data error of 2–5 %.

In [ ]:
d_ana = -pytem.fwd_analytical_central(rho, a, times)
d_dlf = -pytem.fwd_circle_central([], [rho], a, times, transform='dlf')
d_eul = -pytem.fwd_circle_central([], [rho], a, times, transform='euler')

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].loglog(times, d_ana, 'k-', label='Analytical')
axs[0].loglog(times, d_dlf, 'o', mfc='none', label='Numerical (DLF)')
axs[0].loglog(times, d_eul, 'x', label='Numerical (Euler)')
axs[0].set(xlabel='Time [s]', ylabel='-dB/dt [V/m²]')
axs[0].legend()
for d, lab in [(d_dlf, 'DLF'), (d_eul, 'Euler')]:
    axs[1].semilogx(times, 100 * (d - d_ana) / d_ana, label=lab)
axs[1].set(xlabel='Time [s]', ylabel='Relative error [%]')
axs[1].legend()
plt.tight_layout()

### Task 1.2: accuracy vs speed
Time the numerical forward model for three settings: DLF with the default filters, DLF with `hankel_filter='key_201', fourier_filter='key_101'`, and Euler. For each, print the time in ms and the maximum relative error.

Hints: `t0 = time.perf_counter()` before and `time.perf_counter() - t0` after. Call each setting once **before** timing it: the first call compiles the Numba kernels.

Does the longer filter always give a smaller error? Why does speed matter more for an inversion than for one forward model?

In [ ]:
options = [dict(transform='dlf'),
           dict(transform='dlf', hankel_filter='key_201', fourier_filter='key_101'),
           dict(transform='euler')]
for kw in options:
    pytem.fwd_circle_central([], [rho], a, times, **kw)          # compile
    t0 = time.perf_counter()
    d = -pytem.fwd_circle_central([], [rho], a, times, **kw)
    err = np.max(np.abs(d - d_ana) / d_ana) * 100
    print(f'{str(kw):75s} {1e3 * (time.perf_counter() - t0):6.2f} ms   max error {err:.3f} %')

### Task 1.3: the late-time apparent resistivity
`dbdt_to_apprho` converts dB/dt to apparent resistivity using the *late-time* approximation of the half-space solution. Apply it to the exact analytical response and plot apparent resistivity against time, with a dashed line at the true 100 Ohm.m.

Over a uniform half-space it should give 100 Ohm.m at all times. Does it? Where does it fail, and why?

In [ ]:
rho_a = pytem.dbdt_to_apprho(d_ana, np.pi * a**2, times)

plt.semilogx(times, rho_a)
plt.axhline(rho, color='k', ls='--')
plt.xlabel('Time [s]'); plt.ylabel('Apparent resistivity [Ohm.m]')

### Task 1.4 (if time)
Repeat 1.3 for a 10 and a 1000 Ohm.m half-space, and for a 5 m and a 200 m loop radius. When is the late-time approximation valid? Compare with the diffusion depth $\delta \approx \sqrt{2 t \rho / \mu_0}$ relative to the loop size.

In [ ]:
fig, ax = plt.subplots()
for r in [10, 100, 1000]:
    for ai in [5, 200]:
        d = -pytem.fwd_analytical_central(r, ai, times)
        ax.semilogx(times, pytem.dbdt_to_apprho(d, np.pi * ai**2, times) / r, label=f'{r} Ohm.m, a = {ai} m')
ax.axhline(1, color='k', ls='--')
ax.set(xlabel='Time [s]', ylabel='Apparent / true resistivity', ylim=(0.5, 5), yscale='log')
ax.legend(fontsize=8)